In [ ]:
from glob import glob
from cmcmultimodal.core.io import mat2nii
from pathlib import Path
import os

my_files = glob('/Users/Vasilis/Downloads/Zebel/PSOCT/Retardance/Slice*.mat')
highres_files = []
lowres_files = []
for f in my_files:
    nii_highres, nii_lowres = mat2nii(f, nii_lowres_file=os.path.join(Path(f).parent,'lowres/',
                                      Path(f).name.replace('.mat','.nii.gz')), downsample=10)
    highres_files.append(nii_highres)
    lowres_files.append(nii_lowres)
    
highres_files

In [ ]:
#!/bin/bash

python psoct_wrapper.py --inp_path '/Users/Vasilis/CMC_data/Moe' --out_path '/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret' --seq_params '/Users/Vasilis/CMC_data/Moe/PSOCT/seq_params.json' --mri_ref '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz' --reg_modality Retardance --reg_method 'cc' --other_images Retardance Orientation --bad_slides 140 --verbose

In [3]:
# Process all MOE slides

from cmcmultimodal.core.proc import psoct
from pathlib import Path

datadir = '/Users/Vasilis/CMC_data/Moe'
output_path = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_test'
mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz'
seq_params = '/Users/Vasilis/CMC_data/Moe/PSOCT/seq_params.json'

my_data = psoct(Path(datadir), seq_params, reg_modality='Cross', verbose=True)#,
                # slide_range = [60, 100])

indiv_slides = my_data.run_pipeline(other_images=[],#'Cross', 'Retardance'], 
                                    output_path=output_path, 
                                    mri_ref=mri_ref, 
                                    downsample=1, 
                                    bad_slides=[],#140,],
                                    reg_method='flirt')


Reading input information for '/Users/Vasilis/CMC_data/Moe' ...
	Input folder read successfully.
	PSOCT sequence parameters read successfully.
	Found 8 missing slides: [99, 100, 229, 105, 106, 107, 108, 109]

Starting slide registration process ...
	Missing slides have been interpolated successfully.
	Reference slide for alignment: 184, size=(1100, 874)
Slide registration completed.

Starting slide deck creation ...
	Relative and absolute transformation matrices saved.
	Creating slide deck image ... Done.
	Running MRI-to-PSOCT registration ... Done.
	Running PSOCT-to-MRI registration ... Done.
	Updating headers for registration slides ...  Done.

PSOCT pipeline completed and results saved to /Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_test


In [ ]:
from pathlib import Path
from cmcmultimodal.tests.validate_results import compare_results_folder

ref_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret')
est_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret_new')

compare_results_folder(ref_path, est_path, subdir=True)

- Orientation : File does not exist in estimated path!


In [3]:
from fsl.wrappers import fnirt
import subprocess

mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz'
deck = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/Cro_slide_deck.nii.gz'
outfile = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/Cro_slide_deck_in_MRI_nl'
affmat = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/PSOCT_to_MRI.mat'
config = '../data/fnirt_config'

fnirt(src=deck, ref=mri_ref, iout=outfile, aff=affmat, config=config, verbose=True)

# subprocess.run(["fnirt",
#                 f"--in={deck}",
#                 f"--iout={outfile}",
#                 f"--ref={mri_ref}",
#                 f"--aff={affmat}",
#                 f"--config={config}",
#                 "--verbose",
# ])


New Lambda: 300
New FWHM (mm) for --ref: 4
New FWHM (mm) for --in: 4
New Matrix Size: 45 45 15
New Voxel Size: 3.23699 3.23699 3.20001
SSD = 1242.37	n = 2688	Disp reg = 0	Int reg = 13.9824	Total Cost = 1256.35
SSD = 136.997	n = 2692	Disp reg = 0.185544	Int reg = 8.46874	Total Cost = 145.652
SSD = 71.4158	n = 2714	Disp reg = 2.09385	Int reg = 5.41154	Total Cost = 78.9212
SSD = 277.422	n = 2724	Disp reg = 21.4481	Int reg = 2.9987	Total Cost = 301.869
SSD = 49.5498	n = 2723	Disp reg = 3.59108	Int reg = 5.15654	Total Cost = 58.2974
SSD = 225.542	n = 2744	Disp reg = 27.1994	Int reg = 2.71592	Total Cost = 255.457
SSD = 44.0145	n = 2744	Disp reg = 5.03258	Int reg = 4.6597	Total Cost = 53.7068
SSD = 171.837	n = 2747	Disp reg = 25.6378	Int reg = 2.81789	Total Cost = 200.292
SSD = 40.5928	n = 2746	Disp reg = 5.75207	Int reg = 4.53954	Total Cost = 50.8844
Jacobian range is 0.837285 -- 1.24548
***Going to next resolution level***
New FWHM (mm) for --ref: 3
New FWHM (mm) for --in: 3
Setting subsamp

{}

In [ ]:
from fsl.wrappers.avwutils  import fslmerge
import glob


out_file = '/Users/Vasilis/CMC_results/Moe_flirt_Cross/Ret_slide_deck_in_mri_2'
inp_files = sorted(glob.glob('/Users/Vasilis/CMC_results/Moe_flirt_Cross/Retardance/lowres/Slice*'), reverse=True)

fslmerge('y', out_file, *inp_files[98:246])

{}

In [ ]:
import json
import numpy as np
from pathlib import Path

output_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_test')

with open(output_path / "abs_mat.json") as f:
    data = json.load(f)

abs_mat = {int(k): np.array(v) for k, v in data.items()}